# Tutorial 2: DCF Valuation

Run a Discounted Cash Flow (DCF) valuation with sensitivity analysis and Excel export.

**What you'll learn:**
- How to run the DCF pipeline
- How to access DCFResult fields (WACC, implied price, sensitivity table)
- How to export results to Excel

In [ ]:
from finrobot import FinRobot

agent = FinRobot(model="deepseek:deepseek-chat")

## Run DCF Valuation

The DCF pipeline:
1. Fetches historical financials
2. Computes WACC (cost of capital)
3. Projects free cash flows
4. Calculates terminal value
5. Derives implied share price
6. Builds sensitivity table (WACC vs terminal growth rate)

In [ ]:
result = await agent.adcf("AAPL")
print(result.format_summary())

## Access Structured DCF Results

In [ ]:
from finrobot.engine.models.financial import DCFResult

# Get the typed DCFResult from structured_data
dcf: DCFResult | None = None
for val in result.structured_data.values():
    if isinstance(val, DCFResult):
        dcf = val
        break

if dcf:
    print(f"WACC:           {dcf.wacc:.2%}")
    print(f"Implied Price:  ${dcf.implied_price:.2f}")
    print(f"Enterprise Val: ${dcf.enterprise_value / 1e9:.1f}B")
    print(f"Equity Value:   ${dcf.equity_value / 1e9:.1f}B")

## Sensitivity Analysis Table

The sensitivity table shows implied prices across different WACC and terminal growth rate assumptions.

In [ ]:
if dcf and dcf.sensitivity_table:
    st = dcf.sensitivity_table
    # Print header
    tg_header = "WACC \\ TG  | " + " | ".join(f"{tg:.1%}" for tg in st["tg_values"])
    print(tg_header)
    print("-" * len(tg_header))
    # Print rows
    for i, wacc in enumerate(st["wacc_values"]):
        prices = [f"${p:.0f}" if p else "N/A" for p in st["implied_prices"][i]]
        print(f"{wacc:.1%}         | " + " | ".join(prices))

## Export to Excel

In [ ]:
if dcf:
    from finrobot.engine.compute.spreadsheet_gen import generate_dcf_excel

    xlsx_bytes = generate_dcf_excel(dcf, dcf.inputs)
    with open("AAPL_dcf.xlsx", "wb") as f:
        f.write(xlsx_bytes)
    print(f"Saved AAPL_dcf.xlsx ({len(xlsx_bytes):,} bytes)")

In [ ]:
await agent.close()